# 하이퍼파라미터 튜닝

사이킷런(Scikit-Learn)에 내장된 **위스콘신 유방암(Breast Cancer) 데이터 세트**를 활용하여, 대표적인 하이퍼파라미터 튜닝 알고리즘 성능을 비교

In [3]:
import time
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV, RandomizedSearchCV
from sklearn.ensemble import RandomForestClassifier

!pip install optuna
import optuna

# 1. 데이터 로드 및 분할
data = load_breast_cancer()
X_train, X_test, y_train, y_test = train_test_split(
    data.data, data.target, test_size=0.2, random_state=42
)

# 기본 모델 생성
base_model = RandomForestClassifier(random_state=42)

#### [문제 1] Grid Search 구현하기

In [ ]:
param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [5, 10, 15]
}

# [빈칸 1]: Grid Search 객체를 생성하는 클래스명을 채우세요.
grid_search = __________(
    estimator=base_model,
    param_grid=param_grid,
    cv=5,
    scoring='accuracy',
    n_jobs=-1
)

# [빈칸 2]: 학습 및 교차 검증을 시작하는 메서드명을 채우세요.
grid_search.__________(X_train, y_train)

print(f"[Grid Search] 최적 파라미터: {grid_search.best_params_}")
print(f"[Grid Search] 최고 CV 정확도: {grid_search.best_score_:.4f}\n")

#### [문제 2] Random Search 구현하기

In [ ]:
param_dist = {
    'n_estimators': [50, 100, 150, 200, 250],
    'max_depth': [3, 5, 10, 15, None]
}

# [빈칸 3]: 무작위 탐색 시도 횟수를 결정하는 매개변수(Argument) 이름을 채우세요.
random_search = RandomizedSearchCV(
    estimator=base_model,
    param_distributions=param_dist,
    __________=15,
    cv=5,
    scoring='accuracy',
    random_state=42,
    n_jobs=-1
)
random_search.fit(X_train, y_train)

print(f"[Random Search] 최적 파라미터: {random_search.best_params_}")
print(f"[Random Search] 최고 CV 정확도: {random_search.best_score_:.4f}\n")

#### [문제 3] 베이지안 최적화(Optuna) 구현하기

In [ ]:
def objective(trial):
    # [빈칸 4]: 지정한 범위(50~300) 내에서 정수형 파라미터 값을 추론하는 trial 메서드를 채우세요.
    n_estimators = trial.__________('n_estimators', 50, 300)
    max_depth = trial.__________('max_depth', 3, 20)
    
    clf = RandomForestClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        random_state=42,
        n_jobs=-1
    )
    
    score = cross_val_score(clf, X_train, y_train, cv=5, scoring='accuracy').mean()
    return score

# [빈칸 5]: Optuna에서 스터디(최적화) 객체를 생성하는 함수를 채우세요.
study = optuna.__________(direction='maximize')

# [빈칸 6]: 목적 함수(objective)의 최적화를 실행하는 메서드명을 채우세요.
study.__________(objective, n_trials=15)

print(f"[Optuna] 최적 파라미터: {study.best_params}")
print(f"[Optuna] 최고 CV 정확도: {study.best_value:.4f}")

# 머신러닝

유방암 데이터 기반 부스팅(Boosting) 모델 교차검증 및 성능 평가

사이킷런(Scikit-Learn)에 내장된 **위스콘신 유방암(Breast Cancer) 데이터 세트**를 활용하여, 대표적인 부스팅 앙상블 모델들의 성능을 비교

In [ ]:
import pandas as pd
import numpy as np
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings('ignore')

# 1. 유방암 데이터 세트 로드
cancer = load_breast_cancer()
X_features = cancer.data
y_label = cancer.target

print(f"데이터 세트 형태: 피처 {X_features.shape}, 레이블 {y_label.shape}")

# [빈칸 1]: 전체 데이터를 학습용(Train)과 테스트용(Test)으로 분리 (테스트 사이즈 0.2, 난수 시드 156 설정)
# 피처 데이터(X_features)와 레이블 데이터(y_label)를 분할하는 함수를 적어주세요.
X_train, X_test, y_train, y_test = _____(X_features, y_label, test_size=0.2, random_state=156)

print(f"학습 데이터 형태: {X_train.shape}, 테스트 데이터 형태: {X_test.shape}")

### 부스팅 모델 정의
사이킷런의 Gradient Boosting과 널리 쓰이는 XGBoost, LightGBM 라이브러리를 불러와 각각 모델 객체를 생성

In [ ]:
from sklearn.ensemble import GradientBoostingClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

# 2. 3가지 부스팅 모델 객체 생성 (재현성을 위해 random_state 고정)
gb_clf = GradientBoostingClassifier(random_state=156)
xgb_clf = XGBClassifier(n_estimators=100, random_state=156)
lgbm_clf = LGBMClassifier(n_estimators=100, random_state=156)

### 교차 검증 기반 학습 및 평가 함수 작성

각 모델의 객체와 데이터를 입력받아 **K-Fold 교차 검증**을 수행하고, 최종 테스트 데이터에 대한 예측 성능을 반환하는 함수를 작성

In [ ]:
from sklearn.model_selection import KFold
from sklearn.metrics import accuracy_score

# 3. K-Fold 교차검증 기반의 부스팅 평가 함수 정의
def evaluate_boosting_model(model, X_train_n, y_train_n, X_test_n, y_test_n, n_folds):
    # [빈칸 2]: 지정된 n_folds값으로 교차검증 객체 생성.
    kf = _____(n_splits=n_folds, shuffle=True, random_state=156)
    cv_accuracy = []

    print('======================================================')
    print(model.__class__.__name__ , ' model 학습 및 검증 시작 ')
    print('======================================================')

    for folder_counter , (train_index, valid_index) in enumerate(kf.split(X_train_n)):
        # 입력된 학습 데이터에서 부스팅 모델이 학습/검증할 폴드 데이터 셋 추출
        X_tr = X_train_n[train_index]
        y_tr = y_train_n[train_index]
        X_val = X_train_n[valid_index]
        y_val = y_train_n[valid_index]

        # [빈칸 3]: 폴드 세트 내부에서 분리된 학습 데이터(X_tr, y_tr)로 모델 학습 수행
        model._____(X_tr , y_tr)

        # [빈칸 4]: 학습된 모델을 이용해 검증 데이터(X_val) 예측 수행
        pred = model._____(X_val)

        # [빈칸 5]: 실제값(y_val)과 예측값(pred)을 비교해 정확도(Accuracy) 평가 후 리스트에 저장
        accuracy = _____(y_val, pred)
        cv_accuracy.append(accuracy)

        print('\t 폴드 세트 {0} 검증 정확도: {1:.4f}'.format(folder_counter, accuracy))

    # 폴드 세트 내 검증 정확도의 평균을 계산
    print('## 전체 폴드 평균 검증 정확도: {0:.4f}'.format(np.mean(cv_accuracy)))
    print('-'*54)

    # 전체 학습 데이터로 최종 학습 후 테스트 데이터로 최종 성능 평가
    model.fit(X_train_n, y_train_n)
    test_pred = model.predict(X_test_n)
    test_acc = accuracy_score(y_test_n, test_pred)

    return test_acc

### 모델별 실행 및 최종 예측 성능 비교

작성한 함수를 사용하여 **7겹(7-Folds) 교차 검증**을 수행하고, 각 부스팅 모델의 최종 테스트 정확도를 비교

In [ ]:
# 4. 개별 부스팅 모델별 교차검증 평가 수행 (n_folds = 7)
gb_test_acc = evaluate_boosting_model(gb_clf, X_train, y_train, X_test, y_test, 7)
xgb_test_acc = evaluate_boosting_model(xgb_clf, X_train, y_train, X_test, y_test, 7)
lgbm_test_acc = evaluate_boosting_model(lgbm_clf, X_train, y_train, X_test, y_test, 7)

# 최종 결과 종합 출력
print('\n[최종 테스트 데이터 예측 성능 비교]')
print('Gradient Boosting 최종 테스트 예측 정확도: {0:.4f}'.format(gb_test_acc))
print('XGBoost 최종 테스트 예측 정확도: {0:.4f}'.format(xgb_test_acc))
print('LightGBM 최종 테스트 예측 정확도: {0:.4f}'.format(lgbm_test_acc))